In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from scipy.io import arff
import pandas as pd

# Choose one of the .arff files from your environment
arff_file_path1 = '/content/TimeBasedFeatures-Dataset-15s-AllinOne.arff'
# Load the .arff file
data, meta = arff.loadarff(arff_file_path1)

# Convert to pandas DataFrame
df1 = pd.DataFrame(data)
df1.head()


,duration,total_fiat,total_biat,min_fiat,min_biat,max_fiat,max_biat,mean_fiat,mean_biat,flowPktsPerSecond,...,std_flowiat,min_active,mean_active,max_active,std_active,min_idle,mean_idle,max_idle,std_idle,class1
0,9368711.0,16.0,4.0,1564818.0,1549373.0,190205.285714,203290.456522,389822.391917,370323.719754,10.353612,...,267600.198443,1871488.0,1.983656e+06,2195089.0,1.832197e+05,1234883.0,1420565.0,1523088.0,161096.539275,b'CHAT'
1,7340238.0,18.0,4.0,1567554.0,1527893.0,165686.977273,186914.846154,317267.548742,304370.651301,11.580006,...,221462.862028,1491627.0,3.572433e+06,5653239.0,2.942704e+06,1131498.0,1324636.0,1517774.0,273138.379008,b'CHAT'
2,4644225.0,29.0,15.0,1270547.0,1079974.0,165865.178571,195302.130435,329473.126261,300492.588227,11.412022,...,217475.425246,1758922.0,1.758922e+06,1758922.0,0.000000e+00,1079974.0,1079974.0,1079974.0,0.000000,b'CHAT'
3,4978735.0,19.0,8.0,2492050.0,2457286.0,239543.250000,276596.388889,612435.304238,628339.573544,8.034169,...,436959.716436,1710925.0,2.382905e+06,3054885.0,9.503232e+05,1346073.0,1894031.5,2441990.0,774930.342317,b'CHAT'
4,11838189.0,19.0,10.0,3094089.0,3093543.0,243766.500000,295954.725000,599721.781709,625632.703972,7.602514,...,436129.639296,1747431.0,2.400446e+06,3240696.0,6.232744e+05,1394455.0,1983227.0,3042717.0,725987.829075,b'CHAT'


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb
import warnings

In [ ]:
# 1. Create the filtered dataframe correctly
df_filtered = df1.copy()
df_filtered.replace([np.inf, -np.inf], np.nan, inplace=True)
df_filtered.dropna(inplace=True)

# 2. Create the target (noting the byte string format from your screenshot)
df_filtered['target'] = df_filtered['class1'].apply(lambda x: 1 if x == b'P2P' else 0)

# 3. Define y
y = df_filtered['target']

In [ ]:
X = df_filtered.drop(['class1', 'target'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=50, max_leaf_nodes=32, random_state=42)
lgb_model = lgb.LGBMClassifier(n_estimators=50, num_leaves=16, random_state=42)

In [ ]:
rf_model.fit(X_train, y_train)

RandomForestClassifier(max_leaf_nodes=32, n_estimators=50, random_state=42)

In [ ]:
lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 1549, number of negative: 13457
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001784 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5865
[LightGBM] [Info] Number of data points in the train set: 15006, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.103225 -> initscore=-2.161890
[LightGBM] [Info] Start training from score -2.161890


LGBMClassifier(n_estimators=50, num_leaves=16, random_state=42)

In [ ]:
rf_pred = rf_model.predict(X_test)
lgb_pred = lgb_model.predict(X_test)

# Calculate F1-Score (A metric that balances False Positives and False Negatives)
rf_f1 = f1_score(y_test, rf_pred)
lgb_f1 = f1_score(y_test, lgb_pred)

print("\n--- Final Exam Accuracy (F1-Score) ---")
print(f"Random Forest: {rf_f1:.4f}")
print(f"LightGBM:      {lgb_f1:.4f}")


--- Final Exam Accuracy (F1-Score) ---
Random Forest: 0.7659
LightGBM:      0.8301


In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nClass distribution:")
print(df['class1'].value_counts())
print("\nFirst 3 rows:")
print(df.head(3))

Shape: (18758, 25)

Column names:
['duration', 'total_fiat', 'total_biat', 'min_fiat', 'min_biat', 'max_fiat', 'max_biat', 'mean_fiat', 'mean_biat', 'flowPktsPerSecond', 'flowBytesPerSecond', 'min_flowiat', 'max_flowiat', 'mean_flowiat', 'std_flowiat', 'min_active', 'mean_active', 'max_active', 'std_active', 'min_idle', 'mean_idle', 'max_idle', 'std_idle', 'class1', 'target']

Class distribution:
class1
b'VOIP'         5097
b'BROWSING'     5000
b'FT'           2950
b'CHAT'         2086
b'P2P'          1928
b'STREAMING'     957
b'MAIL'          740
Name: count, dtype: int64

First 3 rows:
    duration  total_fiat  total_biat   min_fiat   min_biat       max_fiat  \
0  9368711.0        16.0         4.0  1564818.0  1549373.0  190205.285714   
1  7340238.0        18.0         4.0  1567554.0  1527893.0  165686.977273   
2  4644225.0        29.0        15.0  1270547.0  1079974.0  165865.178571   

        max_biat      mean_fiat      mean_biat  flowPktsPerSecond  ...  \
0  203290.456522  3898

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

# ── 1. FIX TARGET: multiclass, not binary ──────────────────────────────────
le = LabelEncoder()
df1['target_mc'] = le.fit_transform(df1['class1'])
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# ── 2. ADD YOUR CONTRIBUTION: Directional Asymmetry Ratio ─────────────────
# DAR = mean forward IAT / mean backward IAT
# P2P is symmetric (DAR ≈ 1), File Transfer is asymmetric (DAR >> 1 or << 1)
df1['DAR'] = df1['mean_fiat'] / (df1['mean_biat'] + 1e-9)  # epsilon avoids div/0

# ── 3. DEFINE FEATURE SETS ────────────────────────────────────────────────
base_features = ['mean_fiat', 'mean_biat', 'min_fiat', 'min_biat',
                 'max_fiat', 'max_biat', 'flowPktsPerSecond', 'flowBytesPerSecond']

full_features = base_features + ['DAR']

y = df1['target_mc']

# ── 4. TRAIN ALL 4 CONDITIONS ─────────────────────────────────────────────
results = {}

for feat_name, feats in [('base', base_features), ('with_DAR', full_features)]:
    X = df1[feats]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    for model_name, model in [
        ('RF',  RandomForestClassifier(n_estimators=50, max_leaf_nodes=32, random_state=42, n_jobs=-1)),
        ('LGB', lgb.LGBMClassifier(n_estimators=50, num_leaves=16, random_state=42, verbose=-1))
    ]:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        # Per-class F1
        f1_per_class = f1_score(y_test, pred, average=None)
        macro_f1 = f1_score(y_test, pred, average='macro')

        # TCAM estimate
        if model_name == 'RF':
            total_leaves = sum(e.tree_.n_leaves for e in model.estimators_)
        else:
            total_leaves = model.n_estimators * model.num_leaves

        key = f"{model_name}_{feat_name}"
        results[key] = {
            'macro_f1': macro_f1,
            'f1_per_class': f1_per_class,
            'tcam_entries': total_leaves,
            'model': model,
            'y_test': y_test,
            'pred': pred
        }
        print(f"\n[{key}] Macro F1: {macro_f1:.4f} | TCAM est: {total_leaves}")

# ── 5. PRINT YOUR 2x2 TABLE ───────────────────────────────────────────────
print("\n\n===== YOUR PAPER TABLE =====")
class_names = [c.decode() for c in le.classes_]
p2p_idx  = list(le.classes_).index(b'P2P')
chat_idx = list(le.classes_).index(b'CHAT')

print(f"{'Condition':<20} {'Macro F1':>9} {'P2P F1':>9} {'Chat F1':>9} {'TCAM entries':>14}")
print("-" * 65)
for key, r in results.items():
    print(f"{key:<20} {r['macro_f1']:>9.4f} "
          f"{r['f1_per_class'][p2p_idx]:>9.4f} "
          f"{r['f1_per_class'][chat_idx]:>9.4f} "
          f"{r['tcam_entries']:>14}")

Class mapping: {b'BROWSING': np.int64(0), b'CHAT': np.int64(1), b'FT': np.int64(2), b'MAIL': np.int64(3), b'P2P': np.int64(4), b'STREAMING': np.int64(5), b'VOIP': np.int64(6)}

[RF_base] Macro F1: 0.6860 | TCAM est: 1600

[LGB_base] Macro F1: 0.8077 | TCAM est: 800

[RF_with_DAR] Macro F1: 0.6886 | TCAM est: 1600

[LGB_with_DAR] Macro F1: 0.8207 | TCAM est: 800


===== YOUR PAPER TABLE =====
Condition             Macro F1    P2P F1   Chat F1   TCAM entries
-----------------------------------------------------------------
RF_base                 0.6860    0.7266    0.4504           1600
LGB_base                0.8077    0.8257    0.6354            800
RF_with_DAR             0.6886    0.7333    0.4571           1600
LGB_with_DAR            0.8207    0.8339    0.6793            800


In [ ]:
for key in ['RF_base', 'LGB_with_DAR']:
    print(f"\n===== {key} =====")
    print(classification_report(
        results[key]['y_test'],
        results[key]['pred'],
        target_names=[c.decode() for c in le.classes_]
    ))



===== RF_base =====
              precision    recall  f1-score   support

    BROWSING       0.73      0.94      0.82      1000
        CHAT       0.66      0.34      0.45       417
          FT       0.69      0.59      0.63       590
        MAIL       0.69      0.59      0.64       148
         P2P       0.68      0.78      0.73       386
   STREAMING       0.72      0.47      0.57       191
        VOIP       0.96      0.97      0.96      1020

    accuracy                           0.77      3752
   macro avg       0.73      0.67      0.69      3752
weighted avg       0.77      0.77      0.76      3752


===== LGB_with_DAR =====
              precision    recall  f1-score   support

    BROWSING       0.84      0.94      0.89      1000
        CHAT       0.78      0.60      0.68       417
          FT       0.78      0.73      0.75       590
        MAIL       0.78      0.88      0.83       148
         P2P       0.79      0.88      0.83       386
   STREAMING       0.85      0.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ── 1. Target Encoding (7-class Setup) ────────────────────────────────────
le = LabelEncoder()
df1['target_mc'] = le.fit_transform(df1['class1'])

# ── 2. DAR Feature Definition ─────────────────────────────────────────────
df1['DAR'] = df1['mean_fiat'] / (df1['mean_biat'] + 1e-9)

# ── 3. Feature Configurations ─────────────────────────────────────────────
base_features = ['mean_fiat', 'mean_biat', 'min_fiat', 'min_biat',
                 'max_fiat', 'max_biat', 'flowPktsPerSecond', 'flowBytesPerSecond']
full_features = base_features + ['DAR']
y = df1['target_mc']

# Helper to count TCAM entries (total leaf nodes across all trees)
def get_tcam_entries(model, model_name):
    if model_name == 'RF':
        return sum(e.tree_.n_leaves for e in model.estimators_)
    elif model_name == 'LGB':
        df_trees = model.booster_.trees_to_dataframe()
        # Leaf nodes are indicated by having a negative/null feature value (or explicitly flagged as leaf)
        return df_trees[df_trees['decision_type'].isna()].shape[0]
    return 0

# ── 4. Main 2x2 Training & Evaluation ─────────────────────────────────────
results = {}
X_train_base, X_test_base, y_train, y_test = train_test_split(
    df1[base_features], y, test_size=0.2, random_state=42, stratify=y)

X_train_dar, X_test_dar, _, _ = train_test_split(
    df1[full_features], y, test_size=0.2, random_state=42, stratify=y)

configs = [
    ('RF_base', base_features, X_train_base, X_test_base, RandomForestClassifier(n_estimators=50, max_leaf_nodes=32, random_state=42, n_jobs=-1), 'RF'),
    ('LGB_base', base_features, X_train_base, X_test_base, lgb.LGBMClassifier(n_estimators=50, num_leaves=16, random_state=42, verbose=-1), 'LGB'),
    ('RF_with_DAR', full_features, X_train_dar, X_test_dar, RandomForestClassifier(n_estimators=50, max_leaf_nodes=32, random_state=42, n_jobs=-1), 'RF'),
    ('LGB_with_DAR', full_features, X_train_dar, X_test_dar, lgb.LGBMClassifier(n_estimators=50, num_leaves=16, random_state=42, verbose=-1), 'LGB')
]

for name, feats, xtr, xte, model, m_type in configs:
    model.fit(xtr, y_train)
    preds = model.predict(xte)

    macro_f1 = f1_score(y_test, preds, average='macro')
    acc = accuracy_score(y_test, preds)
    f1_per_class = f1_score(y_test, preds, average=None)

    # Get metrics specifically for P2P and CHAT
    p2p_idx = list(le.classes_).index(b'P2P')
    chat_idx = list(le.classes_).index(b'CHAT')

    report = classification_report(y_test, preds, output_dict=True)
    chat_prec = report[str(chat_idx)]['precision']
    chat_rec = report[str(chat_idx)]['recall']

    tcam = get_tcam_entries(model, m_type)

    results[name] = {
        'macro_f1': macro_f1,
        'accuracy': acc,
        'p2p_f1': f1_per_class[p2p_idx],
        'chat_f1': f1_per_class[chat_idx],
        'chat_precision': chat_prec,
        'chat_recall': chat_rec,
        'tcam': tcam,
        'preds': preds,
        'model': model
    }

# ── 5. Output Table ───────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T[[
    'macro_f1', 'accuracy', 'p2p_f1', 'chat_f1', 'chat_precision', 'chat_recall', 'tcam'
]]
display(results_df)

In [ ]:
# ── 6. Print Reports & Confusion Matrices ─────────────────────────────────
for key in ['RF_base', 'LGB_with_DAR']:
    print(f"\n{'='*15} {key} Classification Report {'='*15}")
    print(classification_report(
        y_test,
        results[key]['preds'],
        target_names=[c.decode() for c in le.classes_]
    ))

    # Confusion Matrix
    cm = confusion_matrix(y_test, results[key]['preds'])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[c.decode() for c in le.classes_],
                yticklabels=[c.decode() for c in le.classes_])
    plt.title(f'Confusion Matrix: {key}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

### Step 4 & 5: Matched-Budget, Multi-Seed Evaluation, & Evasion Testing
We will now run the matched-budget LightGBM configuration, execute a 10-seed robust statistical analysis (with paired hypothesis testing), and implement the feature-space evasion robustness evaluation against timing dilation, dummy-packet injection, and jitter attacks.

In [ ]:
# ── 7. Matched Budget Run ────────────────────────────────────────────────
lgb_matched = lgb.LGBMClassifier(n_estimators=14, num_leaves=16, random_state=42, verbose=-1)
lgb_matched.fit(X_train_dar, y_train)
preds_matched = lgb_matched.predict(X_test_dar)

macro_f1_matched = f1_score(y_test, preds_matched, average='macro')
tcam_matched = get_tcam_entries(lgb_matched, 'LGB')

print(f"[LGB Matched-Budget (14 rounds)] Macro F1: {macro_f1_matched:.4f} | TCAM entries: {tcam_matched}")

# Plotting F1 vs TCAM Entries
plt.figure(figsize=(8, 5))
for name, res in results.items():
    plt.scatter(res['tcam'], res['macro_f1'], s=100, label=name, zorder=5)
plt.scatter(tcam_matched, macro_f1_matched, s=150, marker='*', color='red', label='LGB_matched_DAR', zorder=6)
plt.title('Trade-off: Macro F1-Score vs TCAM Entries')
plt.xlabel('TCAM Entries (Leaf Nodes)')
plt.ylabel('Macro F1-Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
from scipy.stats import ttest_rel, wilcoxon

# ── 8. Multi-Seed Robustness Run (10 Seeds) ───────────────────────────────
seeds = list(range(42, 52))
seed_results = {'RF_base': [], 'RF_with_DAR': [], 'LGB_base': [], 'LGB_with_DAR': []}

for s in seeds:
    # Split data using the specific seed
    X_tr_b, X_te_b, y_tr, y_te = train_test_split(df1[base_features], y, test_size=0.2, random_state=s, stratify=y)
    X_tr_d, X_te_d, _, _ = train_test_split(df1[full_features], y, test_size=0.2, random_state=s, stratify=y)

    # Instantiate models with the seed
    rf_b = RandomForestClassifier(n_estimators=50, max_leaf_nodes=32, random_state=s, n_jobs=-1)
    rf_d = RandomForestClassifier(n_estimators=50, max_leaf_nodes=32, random_state=s, n_jobs=-1)
    lgb_b = lgb.LGBMClassifier(n_estimators=50, num_leaves=16, random_state=s, verbose=-1)
    lgb_d = lgb.LGBMClassifier(n_estimators=50, num_leaves=16, random_state=s, verbose=-1)

    # Train & Evaluate
    rf_b.fit(X_tr_b, y_tr)
    seed_results['RF_base'].append(f1_score(y_te, rf_b.predict(X_te_b), average='macro'))

    rf_d.fit(X_tr_d, y_tr)
    seed_results['RF_with_DAR'].append(f1_score(y_te, rf_d.predict(X_te_d), average='macro'))

    lgb_b.fit(X_tr_b, y_tr)
    seed_results['LGB_base'].append(f1_score(y_te, lgb_b.predict(X_te_b), average='macro'))

    lgb_d.fit(X_tr_d, y_tr)
    seed_results['LGB_with_DAR'].append(f1_score(y_te, lgb_d.predict(X_te_d), average='macro'))

# Print mean ± std
print("===== 10-SEED STATISTICS (Macro F1) =====")
for k, vals in seed_results.items():
    print(f"{k:<15}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

# Hypothesis Testing
for model_name in ['RF', 'LGB']:
    b_key = f"{model_name}_base"
    d_key = f"{model_name}_with_DAR"
    t_stat, t_pval = ttest_rel(seed_results[d_key], seed_results[b_key])
    w_stat, w_pval = wilcoxon(seed_results[d_key], seed_results[b_key])
    print(f"\n--- {model_name} (DAR vs Base) ---")
    print(f"Paired t-test p-value: {t_pval:.6f}")
    print(f"Wilcoxon p-value:      {w_pval:.6f}")

In [ ]:
# ── 9. Feature-Space Evasion Evaluation ───────────────────────────────────
def apply_evasion_attack(X_input, attack_type, strength):
    X_mod = X_input.copy()

    # Modifying forward-direction features
    fwd_cols = ['mean_fiat', 'min_fiat', 'max_fiat']

    if attack_type == 'A1':  # Timing dilation
        X_mod[fwd_cols] = X_input[fwd_cols] * strength
    elif attack_type == 'A2':  # Dummy-packet injection
        X_mod[fwd_cols] = X_input[fwd_cols] / strength
    elif attack_type == 'A3':  # Jitter
        np.random.seed(42)
        noise = np.random.lognormal(0, strength, size=(X_input.shape[0], len(fwd_cols)))
        X_mod[fwd_cols] = X_input[fwd_cols] * noise

    # Recalculate DAR based on newly perturbed forward IAT features
    if 'DAR' in X_mod.columns:
        X_mod['DAR'] = X_mod['mean_fiat'] / (X_mod['mean_biat'] + 1e-9)

    return X_mod

# Hold-out sets
X_test_base_ev = X_test_base.copy()
X_test_dar_ev = X_test_dar.copy()

attacks = {
    'A1': [1, 2, 4, 8],      # Alpha
    'A2': [1, 2, 3, 4],      # Beta
    'A3': [0.1, 0.3, 0.5]    # Sigma
}

evasion_results = []

for att, strengths in attacks.items():
    for s in strengths:
        # Evaluate LGB Base
        X_att_base = apply_evasion_attack(X_test_base_ev, att, s)
        pred_base = results['LGB_base']['model'].predict(X_att_base)
        f1_base = f1_score(y_test, pred_base, average='macro')

        # Evaluate LGB with DAR
        X_att_dar = apply_evasion_attack(X_test_dar_ev, att, s)
        pred_dar = results['LGB_with_DAR']['model'].predict(X_att_dar)
        f1_dar = f1_score(y_test, pred_dar, average='macro')

        # Per-class evasion rate calculation: percentage of instances changed from baseline correct predictions
        # Let's track overall drop-off in performance
        ev_rate_base = np.mean(pred_base != results['LGB_base']['preds'])
        ev_rate_dar = np.mean(pred_dar != results['LGB_with_DAR']['preds'])

        evasion_results.append({
            'Attack': att,
            'Strength': s,
            'LGB_base_F1': f1_base,
            'LGB_dar_F1': f1_dar,
            'Evasion_base': ev_rate_base,
            'Evasion_dar': ev_rate_dar
        })

evasion_df = pd.DataFrame(evasion_results)
display(evasion_df)

In [ ]:
# ── 10. Robustness Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, att in enumerate(['A1', 'A2', 'A3']):
    sub_df = evasion_df[evasion_df['Attack'] == att]
    axes[i].plot(sub_df['Strength'], sub_df['LGB_base_F1'], marker='o', label='LGB Base', linestyle='--')
    axes[i].plot(sub_df['Strength'], sub_df['LGB_dar_F1'], marker='s', label='LGB with DAR')
    axes[i].set_title(f'Attack {att} Performance Robustness')
    axes[i].set_xlabel('Attack Strength')
    axes[i].set_ylabel('Macro F1-Score')
    axes[i].legend()
    axes[i].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()